<div class="alert alert-block alert-info" >
    <h1 style="text-align:center;font-weight: 20px; color:black;"> Titanic 생존자 예측 - 분석 </h1>
</div>

## Titanic 생존자 예측 경진대회 개요 
- **목표** 
 - 본 Competition은 생존유무가 포함된 타이타닉 탑승객의 학습 데이터 정보를 분석하고, 그 정보를 바탕으로 머신러닝을 진행하여 테스트 데이터의 탑승객 생존유무를 예측함. 
- **배경지식**
 - 타이타닉호는 1912년 4월 10일 영국의 사우스햄프턴을 떠나 미국의 뉴욕으로 향하던 첫 항해 중에 4월 15일 빙산과 충돌하여 침몰하였다. 타이타닉호의 침몰로 1,514명이 사망하였음.

- **Competition Home**: https://softeer.ai/practice?type=DA&page=0

### [Data Files]
- train.csv: 학습용 데이터 (Features, Target) - 891건 
- test.csv:  예측용 데이터 (Features) - 418건
- sample_submission.csv: 제출용 파일

### [Features]
 - Name: 탑승객 이름
 - pclass: Ticket class (티켓클래스) 1= 1st, 2=2nd, 3=3rd
 - Sex: 성별 male, female
 - Age: 나이(세)
 - sibsp: # of siblings/spouses aboard the Titanic(함께 탑승한 형제자매, 배우자 수 총합) 
 - parch: # of parents/children aboard the Titanic(함께 탑승한 부모, 자녀 수 총합)
 - ticket: Ticket Nubmer(티켓 넘버)
 - Fare: 승객운임
 - cabin: Cabin Number(객실 넘버)
 - embarked: Port of Embarkatation (탑승항구) 
   - C = Cherbourg, Q= Queenstown, S=Southampton 

 
 
### [Target]
 - Survived (생존유무): 0 = 사망, 1= 생존
   
### [하이퍼 파라미터]    
    'learning_rate': [0.05, 0.1, 0.14],  #default: 0.1
    'n_estimator': [100,200, 300]}          #default: 100
  
### [평가 지표]
- accuracy

## 이것은 베이스라인 코드입니다.
---
- 아래 제공되는 Baseline Code를 기반으로 자율적으로 분석/모델링을 진행해 보세요!

## 1. Library import & Data load

### 1-1)  Library import

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
# classification models

from sklearn.linear_model import LogisticRegression   #로지스틱 
from sklearn.tree import DecisionTreeClassifier           #결정트리
from sklearn.ensemble import RandomForestClassifier       #랜덤포레스트
from sklearn.neighbors import KNeighborsClassifier        #knn
from sklearn.ensemble import ExtraTreesClassifier      #엑스트라 트리
from xgboost import XGBClassifier               #XGB
from lightgbm import LGBMClassifier              #LGBM

In [3]:
#train & test dataset 분리
from sklearn.model_selection import train_test_split

# accuracy_score 호출 
from sklearn.metrics import accuracy_score

# hyperparameter tuning 
from sklearn.model_selection import GridSearchCV

### 1-2)  Data file load

In [4]:
# train.csv 불러오기. 
train = pd.read_csv('./data/train.csv')
train

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [5]:
# test.csv 불러오기. 
test = pd.read_csv('./data/test.csv')
test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [ ]:
#데이터프레임의 정보 보기 (info이용)
#train 셋


In [ ]:
#데이터프레임의 정보 보기 (info이용)
#test셋

## [Step2] 탐색적 데이터 분석 EDA

### (1)기술적 통계분석

In [ ]:
#기초 통계량 확인 - train dataset  


In [ ]:
#기초 통계량 확인 - test dataset  


### (2) 변수별 분포 시각화 

#### (1) 범주형 변수(카테고리형 변수) 분포 확인

- 빈도분석 
 - 대상 피쳐: Survived, Sex, Pclass, Embarked
 - **seaborn의 countplot()** 사용

#### (2) 연속형 변수의 분포 확인 
- Age, SibSp, Parch,Fare
- histogram을 통해 분포 파악
  - bins: x축의 그룹 수 (default: 10)

## [step3] 데이터 전처리
### (1) 중복값 처리 

### (2) 불필요한 피처 삭제
- 분석 시 사용하지 않을 column 삭제
 - 'PassengerId', 'Name', 'Ticket', 'Cabin'

### (3) 결측치 처리

### (4) 아웃라이어 제거

### (5)  feature, target 설정


### (6) 인코딩

### (7) 스케일링

- StandardScaler 등을 이용하여 스케일링

## [step4] 모델 설계 
 - 모델의 선택 및 학습, 예측, 평가 분석 모델 설계
    - LogisticRregression, DecisionTreeClassifier, KNeighborsClassifier
    - 앙상블 모델 
     - (배깅 기법)RandomForestClassifier, ExtraTreesClassifier,  
     - (부스팅 기법)XGBClassifier, LGBMClassifier

## [step5] 모델 선택 및 적용

In [ ]:
pred_test = <blank> 

## [step6] 결과 파일 생성

In [ ]:
submission = pd.read_csv('./data/sample_submission.csv')
submission

In [ ]:
submission['Survived'] = <blank>
submission

In [ ]:
# submission을 csv 파일로 저장
submission.to_csv('./data/MySubmission.csv', index=False)

**Good Luck**